# 09.6 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/06-fine-tuning/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

## Exercise 1: loss by hand

A review is positive. The model outputs logits (1.2, −0.3) for (negative, positive). Compute the probability of *positive* with softmax and the cross-entropy loss $-\ln p_{\text{true}}$. Then do the same for logits (−1.0, 2.0).

In [ ]:
# your code here

## Setup for exercises 2–4

In [ ]:
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

import torch, time
torch.manual_seed(42)
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("training on", device)

In [ ]:
from download import load_reviews
from sklearn.model_selection import train_test_split

reviews = load_reviews()          # first run downloads one 117 MB file, then cached
train, test = train_test_split(reviews, test_size=0.2, random_state=42, stratify=reviews["label"])
print(f"{len(reviews):,} reviews ({reviews['label'].mean():.0%} positive): "
      f"train {len(train):,} / test {len(test):,}")
reviews[["text", "label"]].head()

In [ ]:
N_TRAIN = 2000          # on a slow laptop CPU, 1000 halves the training time
small = train.groupby("label").sample(N_TRAIN // 2, random_state=42).sample(frac=1, random_state=42)
print(f"fine-tuning on {len(small):,} reviews, testing on {len(test):,}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CHECKPOINT = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def batches(texts, size):
    for i in range(0, len(texts), size):
        yield i, tokenizer(list(texts[i:i + size]), truncation=True, max_length=128,
                           padding=True, return_tensors="pt").to(device)

def train_one_epoch(model, texts, labels, lr=5e-5, batch_size=16):
    optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    labels, losses = torch.tensor(labels), []
    model.train()
    for i, batch in batches(texts, batch_size):
        loss = model(**batch, labels=labels[i:i + batch_size].to(device)).loss
        loss.backward(); optimiser.step(); optimiser.zero_grad()
        losses.append(loss.item())
    return losses

def predict_proba(model, texts, batch_size=64):
    model.eval(); out = []
    with torch.no_grad():
        for _, batch in batches(texts, batch_size):
            out.append(torch.softmax(model(**batch).logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

## Exercise 2: less data

Fine-tune on only 500 reviews (250 of each). How much accuracy do you lose?

In [ ]:
# your code here

## Exercise 3: freeze the body

Freeze every parameter except the classification head (`p.requires_grad = False`) and fine-tune on the 2,000 reviews. Compare time and accuracy with full fine-tuning.

In [ ]:
# your code here

## Exercise 4: where does the fine-tuned model still fail?

Fine-tune on the 2,000 reviews, then print the 5 test reviews it is most confidently wrong about. Are they model mistakes or label mistakes?

In [ ]:
# your code here